In [1]:
import numpy as np
import tensorcircuit as tc

tc.set_backend("jax")
print("OK")

OK


TensorCircuit internamente usa tensor networks

Resolver el modelo de Ising cuántico (TFIM) usando: VQE (Variational Quantum Eigensolver), buscando: los parámetros θ que minimizan la energía

In [18]:
import tensorflow as tf

In [16]:
# zz gate matrix to be utilized
zz = np.kron(tc.gates._z_matrix, tc.gates._z_matrix)
print(zz)

[[ 1.  0.  0.  0.]
 [ 0. -1.  0. -0.]
 [ 0.  0. -1. -0.]
 [ 0. -0. -0.  1.]]


We first design the Hamiltonian energy expectation function with the input as quantum circuit.

In [14]:
def tfi_energy(c: tc.Circuit, j: float = 1.0, h: float = -1.0):
    e = 0.0
    n = c._nqubits
    for i in range(n):
        e += h * c.expectation((tc.gates.x(), [i]))  # <X_i>
    for i in range(n - 1):  # OBC
        e += j * c.expectation(
            (tc.gates.z(), [i]), (tc.gates.z(), [(i + 1) % n])
        )  # <Z_iZ_{i+1}>
    return tc.backend.real(e)

Now we make the quantum function with tetha as input and energy L expectation as output.

In [15]:
def vqe_tfim(param, n, nlayers):
    c = tc.Circuit(n)
    paramc = tc.backend.cast(
        param, tc.dtypestr
    )  # We assume the input param with dtype float64
    for i in range(n):
        c.H(i)  #estado inicial = superposición uniforme
    for j in range(nlayers):
        for i in range(n - 1):
            c.exp1(i, i + 1, unitary=zz, theta=paramc[2 * j, i])
        for i in range(n):
            c.rx(i, theta=paramc[2 * j + 1, i])
    e = tfi_energy(c)
    return e

To train the parameterized circuit, we should utilize the gradient information
with gradient descent

In [19]:
vqe_tfim_vag = tc.backend.jit(
    tc.backend.value_and_grad(vqe_tfim), static_argnums=(1, 2)
)
def train_step_tf(n, nlayers, maxiter=10000):
    param = tf.Variable(
        initial_value=tf.random.normal(
            shape=[nlayers * 2, n], stddev=0.1, dtype=getattr(tf, tc.rdtypestr)
        )
    )
    opt = tf.keras.optimizers.Adam(1e-2)
    for i in range(maxiter):
        e, grad = vqe_tfim_vag(param, n, nlayers)
        opt.apply_gradients([(grad, param)])
        if i % 200 == 0:
            print(e)
    return e


train_step_tf(6, 3, 2000)

tf.Tensor(-5.6221333, shape=(), dtype=float32)
tf.Tensor(-7.16944, shape=(), dtype=float32)
tf.Tensor(-7.222803, shape=(), dtype=float32)
tf.Tensor(-7.223734, shape=(), dtype=float32)
tf.Tensor(-7.2248, shape=(), dtype=float32)
tf.Tensor(-7.227832, shape=(), dtype=float32)
tf.Tensor(-7.2352333, shape=(), dtype=float32)
tf.Tensor(-7.2386217, shape=(), dtype=float32)
tf.Tensor(-7.248332, shape=(), dtype=float32)
tf.Tensor(-7.2858143, shape=(), dtype=float32)


<tf.Tensor: shape=(), dtype=float32, numpy=-7.291903495788574>